# USO DE GRISEARCHCV PARA BUSCAR LOS MEJORES PARAMETROS

## 1 - IMPORTAMOS LIBRERIAS MAS COMUNES

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2 - CARGAMOS DATASET

In [2]:
penguins_df = sns.load_dataset('penguins').dropna()
penguins_df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male
...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,Female
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


## 3 - EDA Y PREPARACIÓN DE DATOS

### 3.1 REVISAMOS LAS DIFERENTES ESPECIES DE PINGUINOS PARA PODER CREAR UN MODELO DE CLASIFICACIÓN

In [3]:
penguins_df['species'].value_counts()

,count
species,
Adelie,146
Gentoo,119
Chinstrap,68


### 3.2 CODIFICACIÓN DE VARIABLES CATEGORICA SPECIES

In [4]:
penguins_df.loc[:,'species'] = penguins_df['species'].map({'Adelie':0,'Gentoo':1,'Chinstrap':2})
penguins_df.loc[:,'sex'] = penguins_df['sex'].map({'Male':0,'Female':1})
penguins_df['species'] = penguins_df['species'].astype(int)
penguins_df['sex'] = penguins_df['sex'].astype(int)


In [5]:
penguins_df.dtypes

,0
species,int64
island,object
bill_length_mm,float64
bill_depth_mm,float64
flipper_length_mm,float64
body_mass_g,float64
sex,int64


## 3.3 IDENTIFICAMOS VARIABLES X y Y

In [6]:
X = penguins_df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']]  # Seleccionar características
y = penguins_df['species']

## 3.4 DIVIDIMOS EL DATASET EN ENTRENAMIENTO Y PRUEBA

In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## 3.5 ESCALAMOS LOS DATOS

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

# 4 BUSCAMOS LOS MEJORES PARAMETROS PARA MI MODELO KNN

## 4.1 IMPORTAMOS LIBREARIAS PARA GRIDSEARCHCV

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

## 4.2 DEFINIR LOS HIPERPARAMETROS A OPTIMIZAR

In [12]:
param_grid = {
    'n_neighbors': np.arange(3, 21, 2),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'chebyshev', 'minkowski'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'leaf_size': [10, 20, 30, 40, 50],
    'p': [1, 2]
}

## 4.3 CREAMOS MODELO CON GRIDSEARCHCV

In [13]:
knn = KNeighborsClassifier()
grid_search = GridSearchCV(knn,param_grid,cv=5,scoring='accuracy',n_jobs=-1)
grid_search.fit(X_train,y_train)
print("Mejores parámetros:", grid_search.best_params_)
print("Mejor precisión:", grid_search.best_score_)

Mejores parámetros: {'algorithm': 'auto', 'leaf_size': 10, 'metric': 'manhattan', 'n_neighbors': np.int64(9), 'p': 1, 'weights': 'distance'}
Mejor precisión: 0.789308176100629


## 4.4 CREAMOS MODELO CON LOS MEJORES PARAMETROS

In [14]:
best_knn = grid_search.best_estimator_
best_knn.fit(X_train,y_train)
y_pred = best_knn.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)
print(f'Accuracy : {accuracy:.2f}')

Accuracy : 0.87
